# Load & Push

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from google.colab import drive

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

In [ ]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="narratives.csv", repo_type="dataset",local_dir=".")
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="labeled_examples.csv", repo_type="dataset",local_dir=".")
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

narratives.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

labeled_examples.csv: 0.00B [00:00, ?B/s]

utils.py: 0.00B [00:00, ?B/s]

In [ ]:
df = pd.read_csv("narratives.csv", index_col=0)
df

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
2260174,ambiguous,Aircraft vectored in within 1NM to final appro...,NaN
2260249,ambiguous,While on short final we received a glideslope ...,NaN
2260370,aircraft,Flying at cruise; FL350; the FO was the PF and...,At cruise; FL350; during level-off; the Captai...
2261277,humanfactors,On Day 0 around XA:30; I forgot to get LAANC a...,NaN
2261317,procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,Extremely strong winds blew us off the LOC whi...
...,...,...,...
2314970,aircraft,On the morning of Day 0; I; an Officer with a ...,NaN
2314971,humanfactors,At approximately XA35 local time on Day 0; Air...,NaN
2314972,humanfactors,At approximately XA:36pm on Day 0; we were ope...,NaN
2314978,humanfactors,I was vectoring one aircraft (Aircraft X) for ...,NaN


In [ ]:
from sklearn.model_selection import train_test_split
from utils import split_set, make_balanced

X, y = split_set(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.4)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, random_state=42, stratify=y_train, test_size=0.3)

In [ ]:
def parse_narrative_list(narratives:list[str]):
  return "\n".join([f"Narrative {idx+1} - '{narratives[idx]}'" for idx in range(len(narratives)) if pd.notna(narratives[idx])])

def make_narrative_list(df:pd.DataFrame):
  res = df[df.columns].copy().apply(
      lambda x: parse_narrative_list(x.to_list()),
      axis=1
  )
  return pd.DataFrame(res, columns=['narratives'])

In [ ]:
from datasets import DatasetDict, Dataset
def make_ds(raw_df, y):
  assert len(y) == len(raw_df)

  nar = make_narrative_list(raw_df)
  nar = nar.join(y)

  assert len(nar) == len(y)

  nar = nar.reset_index().rename(columns={
      'index' : 'acn',
      'narratives' : 'text',
      y.name : 'label'
    }
  )

  return Dataset.from_pandas(nar)

train_ds = make_ds(X_train, y_train)
valid_ds = make_ds(X_valid, y_valid)
test_ds = make_ds(X_test, y_test)

ds = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds
})

ds

DatasetDict({
    train: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 10360
    })
    validation: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 4441
    })
    test: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 9868
    })
})

In [ ]:
y_train.value_counts()

,count
Assessments_Primary_Problem,
humanfactors,3539
aircraft,3474
procedure,1016
ambiguous,835
weather,346
airport,256
environment-nonweatherrelated,250
chartorpublication,141
airspacestructure,135


In [ ]:
from datasets import ClassLabel

class_names = list(y.unique())
class_names.sort()

new_features = ds["train"].features.copy()
new_features["label"] = ClassLabel(names=class_names)

ds = ds.cast(new_features)
print(ds)
ds['train'].features['label']

Casting the dataset:   0%|          | 0/10360 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4441 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9868 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 10360
    })
    validation: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 4441
    })
    test: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 9868
    })
})


ClassLabel(names=['aircraft', 'airport', 'airspacestructure', 'ambiguous', 'atcequipment/navfacility/buildings', 'chartorpublication', 'companypolicy', 'environment-nonweatherrelated', 'equipment/tooling', 'humanfactors', 'incorrect/notinstalled/unavailablepart', 'logbookentry', 'manuals', 'mel', 'procedure', 'softwareandautomation', 'staffing', 'weather'])

In [ ]:
ds.push_to_hub("sookiemonster/asrs-narratives")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  93%|#########3| 9.10MB / 9.74MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  98%|#########8| 9.20MB / 9.38MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/sookiemonster/asrs-narratives/commit/c28dc622b60e08e3ae104ad72e0decaab9651ae1', commit_message='Upload dataset', commit_description='', oid='c28dc622b60e08e3ae104ad72e0decaab9651ae1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sookiemonster/asrs-narratives', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sookiemonster/asrs-narratives'), pr_revision=None, pr_num=None)